# Import Library

In [ ]:
import os
import re
import joblib
import numpy as np
import pandas as pd

from nltk.tokenize import word_tokenize

# Config

In [ ]:
SCENARIO = 1

TRAIN_SIZE = "10k" # seusaikan dengan dataset yang dipakai

"""
SCENARIO

1 : TF-IDF + NB
2 : TF-IDF + SVM
3 : FastText + SVM
4 : IndoBERT
"""

In [ ]:
CONFIG = {

    1:{
        "name":"tfidf_nb",
        "vectorizer":"models/tfidf_nb_vectorizer.joblib",
        "model":"models/tfidf_nb.joblib"
    },

    2:{
        "name":"tfidf_svm",
        "vectorizer":"models/tfidf_svm_vectorizer.joblib",
        "model":"models/tfidf_svm.joblib"
    },

    3:{
        "name":"fasttext_svm",
        "embedding":"models/fasttext_model.joblib",
        "model":"models/fasttext_svm.joblib"
    },

    4:{
        "name":"indobert",
        "model_dir":f"models/indobert_{TRAIN_SIZE}"
    }

}

# Load Data & Label Encoder

In [ ]:
df = pd.read_csv(
    "dataset/unseen.csv"
)

df.head()

In [ ]:
def preprocess_text(text):

    text = str(text).lower()

    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )

    text = re.sub(
        r"@\w+",
        " ",
        text
    )

    text = re.sub(
        r"#",
        "",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

In [ ]:
df["text_clean"] = df["text"].apply(
    preprocess_text
)

In [ ]:
le = joblib.load(
    "preprocessed/label_encoder.pkl"
)

# Scenario 1

In [ ]:
if SCENARIO == 1:

    vectorizer = joblib.load(
        CONFIG[1]["vectorizer"]
    )

    model = joblib.load(
        CONFIG[1]["model"]
    )

    X = vectorizer.transform(
        df["text_clean"]
    )

    pred = model.predict(X)

# Scenario 2

In [ ]:
if SCENARIO == 2:

    vectorizer = joblib.load(
        CONFIG[2]["vectorizer"]
    )

    model = joblib.load(
        CONFIG[2]["model"]
    )

    X = vectorizer.transform(
        df["text_clean"]
    )

    pred = model.predict(X)

# Scenario 3

In [ ]:
def sentence_vector_fasttext(
    tokens,
    model,
    vector_size=300
):

    vectors=[]

    for token in tokens:

        if token in model.wv:

            vectors.append(
                model.wv[token]
            )

    if len(vectors)==0:

        return np.zeros(
            vector_size
        )

    return np.mean(
        vectors,
        axis=0
    )

In [ ]:
if SCENARIO == 3:

    ft = joblib.load(
        CONFIG[3]["embedding"]
    )

    svm = joblib.load(
        CONFIG[3]["model"]
    )

    tokens=[

        word_tokenize(
            x
        )

        for x in df["text_clean"]

    ]

    X=np.array([

        sentence_vector_fasttext(
            t,
            ft
        )

        for t in tokens

    ])

    pred=svm.predict(X)

# Scenario 4

In [ ]:
if SCENARIO == 4:

    import torch

    from transformers import (

        AutoTokenizer,

        AutoModelForSequenceClassification

    )

    MODEL_PATH=CONFIG[4]["model_dir"]

    tokenizer=AutoTokenizer.from_pretrained(
        MODEL_PATH
    )

    model=AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH
    )

    pred=[]

    for text in df["text_clean"]:

        inputs=tokenizer(

            text,

            return_tensors="pt",

            truncation=True,

            padding=True,

            max_length=128

        )

        outputs=model(
            **inputs
        )

        label=torch.argmax(
            outputs.logits,
            dim=1
        )

        pred.append(
            label.item()
        )

    pred=np.array(pred)

# Result

In [ ]:
df["prediction"] = le.inverse_transform(
    pred
)

In [ ]:
df[
    [
        "text",
        "prediction"
    ]
]

In [ ]:
df

In [ ]:
os.makedirs(
    "prediction",
    exist_ok=True
)

MODEL_NAME = CONFIG[
    SCENARIO
]["name"]

OUTPUT = f"prediction/{MODEL_NAME}_{TRAIN_SIZE}.csv"

df.to_csv(
    OUTPUT,
    index=False
)

print(
    "Saved:",
    OUTPUT
)